# Flatand Graph Data Import Pipeline

## Approach Sketch
To be implemented...
- one node per outgoing edge, label with where it comes from
- look at switches: at fixed distance which 2 are closest -> suggest using GeoPandas API for this.
- add actions as edge attributes (LFR disjointly to all outgoing)
- export enhanced osmnx 
- write importer for enhanced osmnx to GraphRailEnv -> describe what we expect as input, which attributes etc.
- timetable for graph env
- unit test with policy reaching target (possibly hard-coded actions)
- snaphost-visualization (preferrably one osmnx export with railway features -> could be a moving house or roundabout if there is no train feature...)


## Documentation
### OSMnx and GeoPandas API documentation
* https://osmnx.readthedocs.io/en/stable/user-reference.html
* https://github.com/gboeing/osmnx-examples
* https://geopandas.org/en/stable/docs/user_guide.html

### OSM Railway documentation
* https://wiki.openstreetmap.org/wiki/Railways
* https://wiki.openstreetmap.org/wiki/Railway_stations
* https://wiki.openstreetmap.org/wiki/Tag:railway=signal
* https://wiki.openstreetmap.org/wiki/Tag:railway=switch
* https://wiki.openstreetmap.org/wiki/Tag:railway=rail
* https://wiki.openstreetmap.org/wiki/Tag:route%3Drailway


### OpenRailwayMap -> inspect Railway features
* https://openrailwaymap.app/#view=18.86/47.1194931/9.3132586&style=signals

### Insepct OSM by Node ID / Way ID
* https://www.openstreetmap.org/node/2048248546/history/4
* https://www.openstreetmap.org/way/194299504

In [ ]:
# all dependencies of this notebook (inline for now, to be replaced when refactoring into a data pipeline):
# !python -m pip install jupyter osmnx matplotlib "folium>=0.12" mapclassify "ipyleaflet>=0.19" "flatland-rl @ git+https://github.com/flatland-association/flatland-rl.git@526-edge-lengths"
import osmnx as ox

ox.settings.log_console = True
ox.__version__

In [ ]:
# ox.settings.overpass_url='http://localhost:12347'
# ox.settings.overpass_rate_limit=False

In [ ]:
place = "Walenstadt"
# tiles = "cartodbpositron"
mk = {"radius": 6}
# cols = ["height", "addr:housenumber", "addr:street", "addr:postcode"]

## Download OSMnx graph

`ox.settings.all_oneway` controls how OSM ways become directed edges (osmnx 2.1.1, `graph.py`):

| | Default (`False`) | `all_oneway=True` |
|---|---|---|
| Ways not tagged `oneway` (most railway tracks) | Added in both directions, `(u,v)` and `(v,u)`. The reverse copy gets `reversed=True`. | Added once, in the OSM way's node order. |
| Ways with `oneway=-1` | Added once, reversed. | Added once, reversed. |
| `oneway` edge attribute | Overwritten with a bool `True`/`False`. | Not set by osmnx: the original OSM string stays, or is missing if the way has none. |

osmnx documents `all_oneway=True` as meant only for exporting via `save_graph_xml`. This notebook sets it below anyway, so every way becomes a single directed edge in OSM node order.

In [ ]:
ox.settings.all_oneway = True
G = ox.graph.graph_from_place(
   place,
   retain_all=False, # If True, return the entire graph even if it is not connected. If False, retain only the largest weakly connected component.
   truncate_by_edge=True, # If True, retain nodes the outside bounding box if at least one of the node’s neighbors lies within the bounding box.
   simplify=True, # If True, simplify graph topology with the simplify_graph function.
   custom_filter='["railway"~"rail"]',
)
fig, ax = ox.plot.plot_graph(G, node_size=0, edge_color="w", edge_linewidth=0.2)

## Switch-switch edges: recover one discarded intermediate node

`simplify_graph` collapses every chain of interstitial (non-intersection) nodes
into a single edge between the two real intersections/dead-ends it connects. If
two switches end up directly adjacent in the simplified graph, any interstitial
nodes that originally separated them (degree-2 vertices just shaping the curve)
were silently discarded - we only kept their coordinates in the edge's `geometry`.
A direct switch-to-switch edge is exactly the case our doubled-graph construction
(see the GGG sections below) can't always represent consistently (the crossover and
triangular-junction cases). If such an edge actually had an interstitial node in the
non-simplified graph, keeping just one of them back turns the direct switch-switch
adjacency into switch-plain-switch, which is always representable.

In [ ]:
# a node's OSM railway=switch tag can lag behind the actual topology (see the
# switch_attr_low_degree check earlier in the pipeline) - a true switch is one
# with in+out degree > 2, so use that instead of the raw tag
switches_in_G = {n for n in G.nodes if G.in_degree(n) + G.out_degree(n) > 2}
switch_switch_edges = [
    (u, v) for u, v in G.edges()
    if u in switches_in_G and v in switches_in_G
]
print(f"found {len(switch_switch_edges)} switch-switch edge(s) in the simplified graph:")
for u, v in switch_switch_edges:
    print(f"  {u} -> {v}")


In [ ]:
# same download parameters as above, but unsimplified, so we can see which
# interstitial nodes simplify_graph removed between any two switches
G_non_simple = ox.graph.graph_from_place(
    place,
    retain_all=False,
    truncate_by_edge=True,
    simplify=False,
    custom_filter='["railway"~"rail"]',
)


In [ ]:
# reuse osmnx's own endpoint/path-building logic (see simplify_graph's source)
# to recover exactly the interstitial path it would collapse between each pair
# of nodes in the unsimplified graph, and check it against our switch-switch edges
from osmnx.simplification import _get_paths_to_simplify

paths_in_non_simple = {
    (path[0], path[-1]): path
    for path in _get_paths_to_simplify(G_non_simple, None, None)
}

for u, v in switch_switch_edges:
    path = paths_in_non_simple.get((u, v)) or paths_in_non_simple.get((v, u))
    if path is None:
        # _get_paths_to_simplify() only yields a path when the edge needs
        # simplifying - if both endpoints are already (true, degree>2)
        # endpoints with nothing interstitial between them, it's a genuinely
        # direct edge in the raw, non-simplified data too
        assert G_non_simple.has_edge(u, v) or G_non_simple.has_edge(v, u), (
            f"edge {u}-{v} not found at all in G_non_simple"
        )
        print(f"{u} -> {v}: 0 intermediate nodes - already directly adjacent in the non-simplified graph too")
        continue
    intermediate = path[1:-1]
    print(f"{u} -> {v}: {len(intermediate)} intermediate node(s) in G_non_simple: {intermediate}")


In [ ]:
# Custom simplification: identical to osmnx.simplification.simplify_graph (see
# its source - _is_endpoint/_get_paths_to_simplify/_build_path determine the
# endpoints and interstitial paths; simplify_graph then aggregates each path's
# edge attributes - summing "length"/"travel_time", keeping a single value
# where all segments agree, else keeping the list of unique values - and
# builds the merged edge's `geometry` from every node's coordinates along the
# path), EXCEPT: whenever a path's two endpoints are both switches and it has
# at least one interstitial node, that node is kept instead of removed, and
# the path is built as two edges (through that node) instead of one - using
# the exact same per-path attribute-aggregation/geometry logic, just applied
# to each half separately.
# NOTE: this does not guarantee an intermediate node survives between every
# pair of switches - if two switches are already directly adjacent in
# G_non_simple (0 interstitial nodes for that path), there is nothing to keep,
# and the switch-switch edge remains direct. Whether that happens is purely a
# property of the OSM data (how finely the way was split), not something this
# function controls.

import itertools
from collections.abc import Callable

import networkx as nx
from shapely import LineString, Point


def simplify_graph_keep_switch_intermediate(
    G: nx.MultiDiGraph,
    switch_osmids: set[int],
    edge_attr_aggs: dict[str, Callable] | None = None,
) -> nx.MultiDiGraph:
    """Like osmnx.simplification.simplify_graph, but keeps one interstitial
    node on every switch-switch path (see note above)."""
    if G.graph.get("simplified"):
        raise ValueError("This graph has already been simplified, cannot simplify it again.")
    if edge_attr_aggs is None:
        edge_attr_aggs = {"length": sum, "travel_time": sum}

    G = G.copy()

    def path_edge_attrs(path: list[int]) -> dict:
        """Aggregate one interstitial path's edge attributes and geometry."""
        path_attributes = {}
        for u, v in itertools.pairwise(path):
            edge_data = next(iter(G.get_edge_data(u, v).values()))
            for attr, value in edge_data.items():
                path_attributes.setdefault(attr, []).append(value)
        for attr_name, attr_values in path_attributes.items():
            if attr_name in edge_attr_aggs:
                path_attributes[attr_name] = edge_attr_aggs[attr_name](attr_values)
            elif len(set(attr_values)) == 1:
                path_attributes[attr_name] = attr_values[0]
            else:
                path_attributes[attr_name] = list(set(attr_values))
        path_attributes["geometry"] = LineString(
            [Point((G.nodes[n]["x"], G.nodes[n]["y"])) for n in path]
        )
        return path_attributes

    all_nodes_to_remove = []
    all_edges_to_add = []

    for path in _get_paths_to_simplify(G, None, None):
        is_switch_switch = path[0] in switch_osmids and path[-1] in switch_osmids
        if is_switch_switch and len(path) > 2:
            # keep exactly one interstitial node - split the path in two there
            mid_idx = len(path) // 2
            mid_node = path[mid_idx]
            for sub_path in (path[: mid_idx + 1], path[mid_idx:]):
                all_edges_to_add.append({
                    "origin": sub_path[0],
                    "destination": sub_path[-1],
                    "attr_dict": path_edge_attrs(sub_path),
                })
            all_nodes_to_remove.extend(n for n in path[1:-1] if n != mid_node)
        else:
            all_edges_to_add.append({
                "origin": path[0],
                "destination": path[-1],
                "attr_dict": path_edge_attrs(path),
            })
            all_nodes_to_remove.extend(path[1:-1])

    for edge in all_edges_to_add:
        G.add_edge(edge["origin"], edge["destination"], **edge["attr_dict"])
    G.remove_nodes_from(set(all_nodes_to_remove))
    G.graph["simplified"] = True
    return G


# same degree-based definition as above (not the raw OSM tag), computed on
# G_non_simple directly - its real branch points still have degree > 2
switch_osmids = {n for n in G_non_simple.nodes if G_non_simple.in_degree(n) + G_non_simple.out_degree(n) > 2}
G_fixed = simplify_graph_keep_switch_intermediate(G_non_simple, switch_osmids)

switches_in_G_fixed = {n for n in G_fixed.nodes if G_fixed.in_degree(n) + G_fixed.out_degree(n) > 2}
fixed_switch_switch_edges = [
    (u, v) for u, v in G_fixed.edges()
    if u in switches_in_G_fixed and v in switches_in_G_fixed
]
print(f"switch-switch edges remaining after the fix: {fixed_switch_switch_edges}")


In [ ]:
G = G_fixed

In [ ]:
gdf = ox.features.features_from_place(place, 
                                      tags={"building": True}
                                     )
m = gdf.explore(
    #tiles=tiles,
    #tooltip=cols
)
nodes,edges = ox.convert.graph_to_gdfs(G)

In [ ]:
edges.explore(m=m, color="purple")

In [ ]:
nodes.explore(marker_kwds={"radius": 8})

In [ ]:
list(G.nodes(data=True))

## Visualize networkx graph

In [ ]:
import networkx as nx
nx.draw(G, labels={n[0]: str(n[0]) for n in list(G.nodes(data=True))})

## Stations

In [ ]:
stations = ox.features.features_from_place(place, tags={"railway": "station"})
stations.explore(m=m, 
                 #tiles=tiles, 
                 color="yellow", tooltip="name", marker_kwds=mk)

## Signals

In [ ]:
signals = ox.features.features_from_place(place, tags={"railway": "signal"})

In [ ]:
m = gdf.explore(
    #tiles=tiles,
    #tooltip=cols
)
nodes,edges = ox.convert.graph_to_gdfs(G)
nodes.explore(m=m, color="purple")

## Export graph to graphml

In [ ]:
ox.io.save_graphml(G, filepath="walensee.graphml")

## Import graph from graphml

In [ ]:
GG = ox.io.load_graphml(filepath="walensee.graphml")

In [ ]:
# MultiDiGraph -> DiGraph would merge parallel edges (two tracks between the same pair of nodes) - make sure we lose none.
num_edges = GG.number_of_edges()
GG = nx.DiGraph(GG)
assert GG.number_of_edges() == num_edges, f"parallel edges lost in DiGraph conversion: {num_edges} -> {GG.number_of_edges()}"

## Graph Rail Env

In [ ]:
for n, d in GG.nodes(data=True):
    if "railway" not in d or d["railway"] != "switch":
        continue
    print(type(n))
    print(n,d)

In [ ]:
SWITCH=2042222936

In [ ]:
for n in list(GG.successors(SWITCH)):
    print(n, GG.get_edge_data(SWITCH, n, default=None))

In [ ]:
TARGET=2042193124

In [ ]:
for n in list(GG.predecessors(SWITCH)):
    print(n, GG.get_edge_data(n, SWITCH, default=None))

In [ ]:
# !python -m pip install "flatland-rl @ git+https://github.com/flatland-association/flatland-rl.git@526-edge-lengths"
from flatland.core.env_observation_builder import DummyObservationBuilder
from flatland.envs.graph.rail_graph_transition_map import GraphTransitionMap
from flatland.core.graph.graph_resource_map import GraphResourceMap
from flatland.envs.graph.distance_map import GraphDistanceMap
from flatland.envs.graph_rail_env import GraphRailEnv
from flatland.envs.timetable_utils import Timetable
from flatland.envs import malfunction_generators as mal_gen
from flatland.envs.rewards import DefaultRewards
from flatland.envs.rail_env import RailEnvActions

In [ ]:
# add actions: all rail env actions on every edge - GG isn't switch-aware (a node's outgoing edges follow the
# OSM ways' drawing direction, not the switch's point/legs), so its actions are not partitioned at switches and dead
# ends; see "add actions to GGG" below for the partitioned actions
for v1,v2, edge_data in GG.edges(data=True):
    edge_data["actions"] = set(RailEnvActions)
    #GG.edges(v1,v2)["actions"]= {RailEnvActions.MOVE_FORWARD}
    # print(v1,v2,edge_data)

In [ ]:
resource_map = {}
for n1, n2 in GG.edges:
    resource_map[n1] = str(sorted([n1,n2]))
# print(resource_map)

agent_waypoints = {
    0: [[SWITCH], [TARGET]]
}
timetable = Timetable(
    earliest_departures=[[0,None]],
    latest_arrivals=[[None,5]],
    max_episode_steps=99,
)

agent_speeds = {
    0: 1/2
}

graph_env = GraphRailEnv.from_graph(
    g=GG,
    resource_map=resource_map,
    agent_waypoints=agent_waypoints,
    agent_speeds=agent_speeds,
    observation_builder=DummyObservationBuilder(),
    malfunction_generator=mal_gen.NoMalfunctionGen(),
    timetable_generator=lambda *args, **kwargs: timetable,
    seed=42,
    rewards=DefaultRewards(),
)
graph_env.reset()

In [ ]:
GG.get_edge_data(SWITCH,TARGET)

In [ ]:
graph_env.rail.g.get_edge_data(SWITCH,TARGET)

In [ ]:
graph_env.reset()
for _ in range(99):
    print("==============================")
    print(f"elapsed before step {graph_env._elapsed_steps}")
    for agent in graph_env.agents:
        print(f"agent.initial_entry_point={agent.initial_entry_point}")
        print(f"agent.current_entry_point={agent.current_entry_point}")
        print(f"agent.targets={agent.targets}")
        print(f"agent.speed_counter.speed={agent.speed_counter.speed}")
        print(f"agent.speed_counter.max_speed={agent.speed_counter.max_speed}")
        print(f"agent.state={agent.state}")
        
    graph_env.step({0: RailEnvActions.MOVE_FORWARD})
    if graph_env.dones["__all__"]:
        print("==============================")
        print("done")
        break

## Switch geometry: which two legs stay closest?

For the switch node `SWITCH` (from the osmnx graph `GG` loaded back from `walensee.graphml`) we walk
`DISTANCE` metres along each of its three incident edges (starting at the switch) and compare the
resulting probe points pairwise: the pair that stays closest together is the one that continues
through the switch in (nearly) the same direction, the remaining leg is the odd one out.

In [ ]:
import itertools

import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString, Point

DISTANCE = 10  # metres along the edge, measured from the switch


def oriented_geometry(graph: nx.MultiDiGraph, u: int, v: int, k: int, start: int) -> LineString:
    """Geometry of edge (u, v, k) as a LineString that starts at node `start`."""
    geom = graph.edges[u, v, k].get("geometry")
    if geom is None:
        # unsimplified graph: the edge is the straight segment between its two nodes
        geom = LineString([(graph.nodes[u]["x"], graph.nodes[u]["y"]),
                           (graph.nodes[v]["x"], graph.nodes[v]["y"])])
    start_point = Point(graph.nodes[start]["x"], graph.nodes[start]["y"])
    if start_point.distance(Point(geom.coords[-1])) < start_point.distance(Point(geom.coords[0])):
        geom = geom.reverse()
    return geom


# GG was downcast to a plain DiGraph above (for the flatland env), but osmnx's
# projection/plotting helpers need a MultiDiGraph - rebuild one just for this section
GG_multi = nx.MultiDiGraph(GG)

# project to UTM so that lengths and interpolation are in metres
G_proj = ox.projection.project_graph(GG_multi)

# all edges incident to the switch, no matter their direction in the digraph
incident = ([(SWITCH, v, k, v) for _, v, k in G_proj.out_edges(SWITCH, keys=True)]
            + [(u, SWITCH, k, u) for u, _, k in G_proj.in_edges(SWITCH, keys=True)])

probes = {}
for u, v, k, neighbour in incident:
    geom = oriented_geometry(G_proj, u, v, k, SWITCH)
    if geom.length < DISTANCE:
        print(f"edge {SWITCH} -> {neighbour} is only {geom.length:.1f} m long, "
              f"probe clipped to its far end")
    probes[neighbour] = geom.interpolate(DISTANCE)

probe_points = gpd.GeoDataFrame({"neighbour": list(probes)},
                                geometry=list(probes.values()),
                                crs=G_proj.graph["crs"])
probe_points

In [ ]:
pairs = pd.DataFrame(
    [(a, b, pa.distance(pb)) for (a, pa), (b, pb) in itertools.combinations(probes.items(), 2)],
    columns=["neighbour_1", "neighbour_2", "distance"],
).sort_values("distance", ignore_index=True)
display(pairs)

closest = pairs.iloc[0]
print(f"at {DISTANCE} m from {SWITCH} the closest pair is "
      f"{closest.neighbour_1} / {closest.neighbour_2} ({closest.distance:.3f} m apart); "
      f"odd one out: {set(probes) - {closest.neighbour_1, closest.neighbour_2}}")

In [ ]:
# visual check: the switch (blue), its three incident edges (green), the three original
# neighbours (orange) and the three probe points at DISTANCE along each edge (red)
ego = nx.ego_graph(G_proj, SWITCH, radius=1, undirected=True)
ego_nodes, ego_edges = ox.convert.graph_to_gdfs(ego)
m_switch = ego_edges.to_crs(epsg=4326).explore(color="green")
# ego_nodes is indexed by osmid (not a column); explore() resets the index itself
# and surfaces it as an "osmid" tooltip column, so don't add that column again here
ego_nodes.loc[[SWITCH]].to_crs(epsg=4326).explore(m=m_switch, color="blue", tooltip="osmid", marker_kwds=mk)
ego_nodes.loc[list(probes)].to_crs(epsg=4326).explore(m=m_switch, color="orange", tooltip="osmid", marker_kwds=mk)
probe_points.to_crs(epsg=4326).explore(m=m_switch, color="red", tooltip="neighbour", marker_kwds=mk)

## Switch geometry as a reusable function

In [ ]:
# `graph` must be a (multi-)digraph projected to a metric CRS (see
# ox.projection.project_graph). If any incident edge is shorter than
# `distance`, the probing distance is truncated to the shortest incident
# edge's length for ALL of the switch's edges - clipping each edge
# independently would put a short edge's probe much closer to the switch
# than the others, which could make it look artificially "closest" to every
# other neighbour.
def switch_probes(graph: nx.MultiDiGraph, switch: int, distance: float = DISTANCE) -> dict[int, Point]:
    """Point on each of `switch`'s incident edges at the (truncated, see note above) probing distance, per neighbour."""
    multi_graph = graph if graph.is_multigraph() else nx.MultiDiGraph(graph)

    incident = ([(switch, v, k, v) for _, v, k in multi_graph.out_edges(switch, keys=True)]
                + [(u, switch, k, u) for u, _, k in multi_graph.in_edges(switch, keys=True)])

    geoms = {neighbour: oriented_geometry(multi_graph, u, v, k, switch) for u, v, k, neighbour in incident}
    probe_distance = min([distance] + [geom.length for geom in geoms.values()])
    return {neighbour: geom.interpolate(probe_distance) for neighbour, geom in geoms.items()}


def closest_neighbors(graph: nx.MultiDiGraph, switch: int, distance: float = DISTANCE) -> tuple[int, int]:
    """Return the two neighbours of `switch` whose edges stay closest together."""
    probes = switch_probes(graph, switch, distance)

    pairs = pd.DataFrame(
        [(a, b, pa.distance(pb)) for (a, pa), (b, pb) in itertools.combinations(probes.items(), 2)],
        columns=["neighbour_1", "neighbour_2", "distance"],
    ).sort_values("distance", ignore_index=True)

    closest = pairs.iloc[0]
    return closest.neighbour_1, closest.neighbour_2


neighbor_1, neighbor_2 = closest_neighbors(G_proj, SWITCH)
print(f"closest neighbours of switch {SWITCH}: {neighbor_1} and {neighbor_2}")


## Switch geometry: forward and diverging leg

In [ ]:
# The probes of switch_probes() all lie on a circle around the switch. Coming from the point, going straight
# through the switch leads to the point's probe mirrored through the switch (its antipode on the circle): the leg
# whose probe is closer to it (i.e. the smaller angle) is the forward leg, the other one diverges. The diverging leg
# branches off to the right if it lies clockwise of the forward leg (seen from the switch, in the projected CRS with
# x east and y north), i.e. it's a right switch - otherwise to the left.
def switch_legs(graph: nx.MultiDiGraph, switch: int, point: int, distance: float = DISTANCE) -> tuple[int, int, bool]:
    """Return `(forward_leg, diverging_leg, diverges_right)` when passing `switch` from `point` into its legs."""
    probes = switch_probes(graph, switch, distance)
    center = Point(graph.nodes[switch]["x"], graph.nodes[switch]["y"])
    straight = Point(2 * center.x - probes[point].x, 2 * center.y - probes[point].y)
    legs = [neighbour for neighbour in probes if neighbour != point]
    assert len(legs) == 2, f"switch {switch} with point {point} has legs {legs}"
    forward, diverging = sorted(legs, key=lambda leg: probes[leg].distance(straight))
    f, d = probes[forward], probes[diverging]
    cross = (f.x - center.x) * (d.y - center.y) - (f.y - center.y) * (d.x - center.x)
    return forward, diverging, cross < 0


undirected = GG.to_undirected()
for switch in [n for n in GG.nodes if GG.in_degree(n) + GG.out_degree(n) > 2]:
    (point,) = set(undirected.neighbors(switch)) - {int(x) for x in closest_neighbors(G_proj, switch)}
    forward, diverging, diverges_right = switch_legs(G_proj, switch, point)
    print(f"switch {switch}: from point {point}, forward leg {forward}, "
          f"diverging leg {diverging} to the {'right' if diverges_right else 'left'}")

## Sanity check: run `closest_neighbors` on every switch

In [ ]:
switches = [n for n in GG.nodes if GG.in_degree(n) + GG.out_degree(n) > 2]
print(f"found {len(switches)} node(s) with in+out degree > 2: {switches}")

missing_switch_attr = [n for n in switches if GG.nodes[n].get("railway") != "switch"]
assert not missing_switch_attr, (
    f"node(s) with in+out degree > 2 but no switch attribute: {missing_switch_attr}"
)
print("OK: all nodes with in+out degree > 2 have the switch attribute")

switch_attr_low_degree = [
    n for n, d in GG.nodes(data=True)
    if d.get("railway") == "switch" and GG.in_degree(n) + GG.out_degree(n) <= 2
]
print(f"node(s) with switch attribute but in+out degree <= 2: {switch_attr_low_degree}")

for switch in switches:
    try:
        n1, n2 = closest_neighbors(G_proj, switch)
        print(f"switch {switch}: closest neighbours are {n1} and {n2}")
    except Exception as e:
        print(f"switch {switch}: closest_neighbors FAILED with {type(e).__name__}: {e}")

## Visual inspection: switches vs. low-degree switch-tagged nodes

Plot the whole graph, highlighting nodes with in+out degree > 2 (the switches used above) in one
color and nodes that carry the `switch` attribute but have degree <= 2 (flagged by the sanity
check above) in another.

In [ ]:
from matplotlib.lines import Line2D

switches_set = set(switches)
switch_attr_low_degree_set = set(switch_attr_low_degree)

default_color = "#999999"
switch_color = "#1f77b4"
low_degree_switch_color = "#d62728"

node_color = [
    low_degree_switch_color if n in switch_attr_low_degree_set
    else switch_color if n in switches_set
    else default_color
    for n in GG_multi.nodes
]

fig, ax = ox.plot.plot_graph(
    GG_multi,
    node_color=node_color,
    node_size=15,
    edge_color="w",
    edge_linewidth=0.5,
    show=False,
    close=False,
)
ax.legend(
    handles=[
        Line2D([], [], marker="o", linestyle="", color=switch_color, label="switch (in+out degree > 2)"),
        Line2D([], [], marker="o", linestyle="", color=low_degree_switch_color, label="switch attribute, in+out degree <= 2"),
        Line2D([], [], marker="o", linestyle="", color=default_color, label="other node"),
    ],
    loc="lower left",
)

In [ ]:
def explore_categorized(graph: nx.MultiDiGraph | nx.DiGraph) -> "folium.Map":
    """Interactive map of `graph`: edges green, nodes colored by switch category."""
    # ox.convert.graph_to_gdfs() needs (u, v, key) edges, i.e. a MultiDiGraph
    multi_graph = graph if graph.is_multigraph() else nx.MultiDiGraph(graph)
    nodes, edges = ox.convert.graph_to_gdfs(multi_graph)

    m = edges.explore(color="green")

    is_switch = nodes.index.isin(switches_set)
    is_low_degree_switch = nodes.index.isin(switch_attr_low_degree_set)
    is_other = ~(is_switch | is_low_degree_switch)

    if is_other.any():
        nodes[is_other].explore(m=m, color=default_color, tooltip="osmid", marker_kwds=mk)
    if is_switch.any():
        nodes[is_switch].explore(m=m, color=switch_color, tooltip="osmid", marker_kwds=mk)
    if is_low_degree_switch.any():
        nodes[is_low_degree_switch].explore(m=m, color=low_degree_switch_color, tooltip="osmid", marker_kwds=mk)

    return m


# same categories as above, but as an interactive map (see "visual check" above)
explore_categorized(GG_multi)


## Sanity check: max degree in the graph

In [ ]:
over_degree = {n: GG.in_degree(n) + GG.out_degree(n) for n in GG.nodes if GG.in_degree(n) + GG.out_degree(n) > 3}
assert not over_degree, f"node(s) with in+out degree > 3: {over_degree}"
print("OK: all nodes in GG have in+out degree <= 3")

## Build GGG: breadth-first traversal, all paths starting at each node

In [ ]:
from collections import deque


# boundary nodes: only incoming or only outgoing edges in GG (not both)
boundary_nodes = {n for n in GG.nodes if (GG.in_degree(n) == 0) != (GG.out_degree(n) == 0)}
print("boundary nodes:", boundary_nodes)


undirected = GG.to_undirected()

# for each switch, the pair of neighbours whose legs continue straight through it
# (see closest_neighbors()) - going directly from one to the other isn't a real
# routing decision, so the BFS below backtracks instead of walking straight through
close_neighbor_pairs = {s: set(closest_neighbors(G_proj, s)) for s in switches_set}

# non-loopy (simple) paths in the undirected version of GG, starting at each boundary node:
# keep extending the path as long as an unvisited neighbor of the current node exists;
# once no such neighbor can be found, the path is done and gets recorded
boundary_paths = []
for b in boundary_nodes:
    paths = []
    queue = deque([[b]])
    while queue:
        path = queue.popleft()
        last = path[-1]
        predecessor = path[-2] if len(path) > 1 else None
        close_pair = close_neighbor_pairs.get(last)
        extended = False
        for neighbor in undirected.neighbors(last):
            if neighbor in path:
                continue
            if close_pair is not None and predecessor in close_pair and neighbor in close_pair:
                # predecessor -> last -> neighbor would pass straight between last's
                # close neighbors - exclude it, backtrack instead
                continue
            queue.append(path + [neighbor])
            extended = True
        if not extended:
            paths.append(path)

    boundary_paths.extend(paths)
    print(b, paths)

# verify: paths are prefix-free
not_prefix_free = [
    p for p in boundary_paths
    if any(p is not q and len(p) <= len(q) and q[:len(p)] == p for q in boundary_paths)
]
assert not not_prefix_free, f"found non-prefix-free path(s) {not_prefix_free}"
print("OK: all boundary-node paths are prefix-free")

# verify: each path ends at another boundary node
bad_paths = [pp for pp in boundary_paths if pp[-1] not in boundary_nodes]
assert not bad_paths, f"path(s) not ending in a boundary node: {bad_paths}"
print("OK: all boundary-node paths end at another boundary node")

In [ ]:
def prime(n: str) -> str:
    """Toggle `n` between its plain and "_prime" identity."""
    if n.endswith("_prime"):
        return n.rstrip("_prime")
    else:
        return f"{n}_prime"


In [ ]:
# if `n` is a switch (3 undirected neighbors), excludes any neighbor forming a
# close, near-collinear "straight-through" pair with `nn` (see
# closest_neighbors()) - i.e. passing from `nn` straight through the switch.
def find_other_neighbor(graph: nx.Graph, n: int, nn: int, G_proj: nx.MultiDiGraph) -> list[int]:
    """Neighbors of `n` in `graph` other than `nn` (see note above for switches)."""
    neighbors = list(graph.neighbors(n))
    candidates = [x for x in neighbors if x != nn]
    if len(neighbors) == 3:
        close_pair = set(closest_neighbors(G_proj, n))
        if nn in close_pair:
            candidates = [x for x in candidates if x not in close_pair]
    if not candidates:
        raise ValueError(f"no neighbor of {n} other than {nn} satisfies the constraints")
    return candidates


In [ ]:
def verify_ggg_paths(ggg: nx.DiGraph, boundary_paths: list[list[int]]) -> None:
    """Assert every boundary_paths path in GG has a corresponding directed path in `ggg`."""

    def find_ggg_edge(x1: int, x2: int) -> tuple[str, str] | None:
        """The (id1, id2) GGG identities - each str(x)/prime(str(x)) - for the
        GG pair (x1, x2); or None if no such edge exists in `ggg`."""
        x1_str, x2_str = str(x1), str(x2)
        for a in (x1_str, prime(x1_str)):
            for b in (x2_str, prime(x2_str)):
                if (a, b) in ggg.edges:
                    return (a, b)
        return None

    def first_missing_edge(path: list[int]) -> tuple[str, tuple[int, int]] | None:
        """First GG pair in `path` with no GGG edge ("no mapping"), or whose
        GGG edge doesn't chain with the previous one ("inconsistent path");
        None if the whole path checks out."""
        ggg_edges = []
        for x1, x2 in zip(path, path[1:]):
            edge = find_ggg_edge(x1, x2)
            if edge is None:
                return ("no mapping", (x1, x2))
            ggg_edges.append(edge)

        for i, ((n1, n2), (n3, n4)) in enumerate(zip(ggg_edges, ggg_edges[1:])):
            if n2 != n3:
                return ("inconsistent path", (path[i + 1], path[i + 2]))

        return None

    missing = {}
    for p in boundary_paths:
        failure = first_missing_edge(p)
        if failure is not None:
            missing[tuple(p)] = failure
    assert not missing, f"first missing edge per boundary path (reason, edge): {missing}"
    print("OK: every boundary path in GG has a corresponding directed path in GGG")


In [ ]:
def verify_edge_duality(ggg: nx.DiGraph) -> None:
    """Assert (n1, n2) in edges <=> (prime(n2), prime(n1)) in edges, for all n1, n2 in `ggg`."""
    edges = set(ggg.edges)
    mismatches = [
        (n1, n2)
        for n1 in ggg.nodes
        for n2 in ggg.nodes
        if ((n1, n2) in edges) != ((prime(n2), prime(n1)) in edges)
    ]
    assert not mismatches, f"edge-reversal duality violated for: {mismatches}"
    print("OK: (n1, n2) in edges <=> (prime(n2), prime(n1)) in edges for all n1, n2 in ggg.nodes")


In [ ]:
# 1st implementation: BFS (open/closed) with inline case analysis per node
# degree (1/2/3), using find_me()/find_other_neighbor() and a postpone
# mechanism for nodes whose neighbors aren't resolvable in GGG yet.
# Known gap: the degree-3/"1 existing neighbor" branch does not check
# close-neighbor pairs (unlike implementations 2 and 3), so it can produce
# an inconsistent GGG in cases where that matters.
def build_ggg_impl1(
    GG: nx.DiGraph,
    close_neighbor_pairs: dict[int, set[int]],
    G_proj: nx.MultiDiGraph,
) -> nx.DiGraph:
    """Build GGG via BFS with inline per-degree case analysis (see note above)."""

    GGG = nx.DiGraph()
    GGG.graph["crs"] = GG.graph.get("crs")
    undirected = GG.to_undirected()

    def add_oriented_edge(u: str, v: str) -> None:
        """Add edge u -> v to GGG, plus its mirror prime(v) -> prime(u)."""
        u, v = str(u), str(v)
        u_id = int(u[:-len("_prime")]) if u.endswith("_prime") else int(u)
        v_id = int(v[:-len("_prime")]) if v.endswith("_prime") else int(v)
        edge_data = GG.get_edge_data(u_id, v_id) or GG.get_edge_data(v_id, u_id) or {}

        GGG.add_node(u, **GG.nodes[u_id])
        GGG.add_node(v, **GG.nodes[v_id])
        assert u != v, f"self-loop edge {u} -> {v}"
        GGG.add_edge(u, v, **edge_data)

        pu, pv = prime(u), prime(v)
        GGG.add_node(pv, **GG.nodes[v_id])
        GGG.add_node(pu, **GG.nodes[u_id])
        assert pv != pu, f"self-loop edge {pv} -> {pu}"
        GGG.add_edge(pv, pu, **edge_data)

    def find_me(en: str, n_str: str) -> str:
        """Which of n_str/prime(n_str) is already connected to en/prime(en) in
        GGG, direction-agnostic (same_variant vs crossing - see impl 2/3)."""
        en, n_str = str(en), str(n_str)
        same_variant = ((en, n_str) in GGG.edges or (n_str, en) in GGG.edges
                         or (prime(en), prime(n_str)) in GGG.edges or (prime(n_str), prime(en)) in GGG.edges)
        crossing = ((en, prime(n_str)) in GGG.edges or (prime(n_str), en) in GGG.edges
                    or (prime(en), n_str) in GGG.edges or (n_str, prime(en)) in GGG.edges)
        if same_variant and not crossing:
            return n_str
        elif crossing and not same_variant:
            return prime(n_str)
        elif same_variant and crossing:
            raise ValueError(f"both identities of {n_str} already connect to {en}/{prime(en)} - genuine conflict")
        else:
            raise ValueError(f"no edge found connecting {en}/{prime(en)} to {n_str}/{prime(n_str)}")

    initial = list(GG.nodes)[0]
    n = initial

    neighbors = list(undirected.neighbors(initial))
    if len(neighbors) == 2:
        n1, n2 = neighbors[0], neighbors[1]
        add_oriented_edge(n, n1)
        add_oriented_edge(n2, n)
    elif len(neighbors) == 3:
        close_pair = set(closest_neighbors(G_proj, n))
        for nn in close_pair:
            # closest_neighbors() can hand back float-typed ids - normalize to int
            # so the GGG node id matches the plain str(n) ids used everywhere else
            add_oriented_edge(n, int(nn))
        for nn in neighbors:
            if nn in close_pair:
                continue
            add_oriented_edge(nn, n)
    else:
        for nn in neighbors:
            add_oriented_edge(n, nn)

    open_ = list(neighbors)
    closed = {initial}

    while len(open_) > 0:
        n = open_[0]
        open_ = open_[1:]
        if n in closed:
            continue

        neighbors = list(undirected.neighbors(n))
        closed.add(n)

        if len(neighbors) == 1:
            # open only contains reachable nodes, so nothing left to do.
            continue

        elif len(neighbors) == 2:
            # open only contains reachable nodes, so one existing and other new
            existing = [x for x in neighbors if x in closed]
            new = [x for x in neighbors if x not in closed]
            if len(existing) == 2:
                nn1, nn2 = neighbors[0], neighbors[1]

                # assert at least one candidate is already in GGG, take one of those
                nnn1_candidates = [str(x) for x in find_other_neighbor(undirected, nn1, n, G_proj)]
                ggg_undirected = GGG.to_undirected()
                nnn1_in_ggg = [x for x in nnn1_candidates if ggg_undirected.has_edge(x, str(nn1))]
                if not nnn1_in_ggg:
                    open_.append(n)
                    closed.remove(n)
                    continue
                nnn1 = nnn1_in_ggg[0]

                nnn2_candidates = [str(x) for x in find_other_neighbor(undirected, nn2, n, G_proj)]
                nnn2_in_ggg = [x for x in nnn2_candidates if ggg_undirected.has_edge(x, str(nn2))]
                if not nnn2_in_ggg:
                    open_.append(n)
                    closed.remove(n)
                    continue
                nnn2 = nnn2_in_ggg[0]

                v1 = find_me(nnn1, str(nn1))
                v2 = find_me(str(nn2), nnn2)

                add_oriented_edge(v1, str(n))
                add_oriented_edge(str(n), v2)
                add_oriented_edge(prime(v2), prime(str(n)))
                add_oriented_edge(prime(str(n)), prime(v1))
                continue
            elif len(existing) == 0:
                raise ValueError(f"node {n} visited with no existing neighbor")

            assert len(existing) == 1, f"expected exactly one closed neighbor of {n}, got {existing}"
            assert len(new) == 1, f"expected exactly one new neighbor of {n}, got {new}"
            existing_neighbor = existing[0]
            new_neighbor = new[0]

            en = str(existing_neighbor)
            n_str = str(n)
            me = find_me(en, n_str)

            add_oriented_edge(me, str(new_neighbor))
            add_oriented_edge(prime(str(new_neighbor)), prime(me))
            open_.append(new_neighbor)

        elif len(neighbors) == 3:
            # assert one neighbor existing two new, find out which ones
            existing = [x for x in neighbors if x in closed]
            new = [x for x in neighbors if x not in closed]
            if len(existing) == 2:
                close_pair = set(closest_neighbors(G_proj, n))
                existing_close = [x for x in existing if x in close_pair]
                n_str = str(n)
                if len(existing_close) == 2:
                    # case 1: 2 existing are both close neighbors
                    me = find_me(str(existing[0]), n_str)
                    assert me == find_me(str(existing[1]), n_str), (
                        f"close-pair legs of switch {n} resolved to different identities"
                    )
                elif len(existing_close) == 1:
                    # case 2: 1 existing is close neighbor, the other not close
                    existing_point = [x for x in existing if x not in close_pair][0]
                    me = find_me(str(existing_point), n_str)
                else:
                    raise ValueError(
                        f"unexpected close_pair/existing combination at switch {n}: "
                        f"existing={existing}, close_pair={close_pair}"
                    )
                continue
            assert len(existing) == 1, f"expected exactly one closed neighbor of {n}, got {existing}"
            assert len(new) == 2, f"expected exactly two new neighbors of {n}, got {new}"
            existing_neighbor = existing[0]
            en = str(existing_neighbor)
            n_str = str(n)
            me = find_me(en, n_str)

            for new_neighbor in new:
                add_oriented_edge(me, str(new_neighbor))
                add_oriented_edge(prime(str(new_neighbor)), prime(me))
                open_.append(new_neighbor)
        else:
            raise ValueError(f"unexpected degree {len(neighbors)} at node {n}")

    return GGG


In [ ]:
#impl1 incomplete
#GGG = build_ggg_impl1(GG, close_neighbor_pairs, G_proj)

In [ ]:
# verify implementation 1 (BFS with inline degree-based case analysis)
#verify_edge_duality(GGG)


In [ ]:
#verify_ggg_paths(GGG, boundary_paths)


## General, order-independent construction of GGG


In [ ]:
# 2nd implementation: general, order-independent construction of GGG from
# GG's undirected version - orient every physical edge exactly once such
# that, at every switch, the close, near-collinear neighbor pair (its
# "legs" - see closest_neighbors()) always ends up on the SAME side (both
# in, or both out) while the third neighbor (the "point") always ends up on
# the OPPOSITE side - that is the only constraint. Non-switch nodes simply
# alternate in/out between their two neighbors. Which side (in vs out) a
# switch/node actually settles on is only decided once one of its edges is
# already oriented (never fixed upfront) - this is what correctly handles a
# switch directly adjacent to another switch (e.g. a crossover), where each
# switch's own close pair may independently want to treat the shared edge
# as a leg: whichever side gets oriented first determines the other via
# this same propagation, instead of the two independent per-switch choices
# conflicting.
#
# Because a switch's two legs always share one role and its point always
# takes the other, there is never an edge from one leg directly to the
# other through the switch - the only way through is point <-> leg (and,
# via the "_prime" mirror, leg_prime <-> point_prime) - so no path can ever
# go over the small angle in the switch (n1 -> s -> n2 for close neighbors
# n1, n2). This also guarantees every boundary_paths path in GG has a
# corresponding directed path in GGG, since boundary_paths already
# excludes leg-to-leg steps using the same close_neighbor_pairs.
def build_ggg_impl2(GG: nx.DiGraph, close_neighbor_pairs: dict[int, set[int]]) -> nx.DiGraph:
    """Build GGG by flood-filling a consistent edge orientation (see note above).

    Raises AssertionError on a genuine topological conflict (e.g. an odd
    cycle of mutually-adjacent switches) instead of silently resolving it.
    """

    GGG = nx.DiGraph()
    GGG.graph["crs"] = GG.graph.get("crs")
    undirected = GG.to_undirected()

    def add_oriented_edge(u: str, v: str) -> None:
        """Add edge u -> v to GGG, plus its mirror prime(v) -> prime(u)."""
        u, v = str(u), str(v)
        u_id = int(u[:-len("_prime")]) if u.endswith("_prime") else int(u)
        v_id = int(v[:-len("_prime")]) if v.endswith("_prime") else int(v)
        edge_data = GG.get_edge_data(u_id, v_id) or GG.get_edge_data(v_id, u_id) or {}

        GGG.add_node(u, **GG.nodes[u_id])
        GGG.add_node(v, **GG.nodes[v_id])
        assert u != v, f"self-loop edge {u} -> {v}"
        GGG.add_edge(u, v, **edge_data)

        pu, pv = prime(u), prime(v)
        GGG.add_node(pv, **GG.nodes[v_id])
        GGG.add_node(pu, **GG.nodes[u_id])
        assert pv != pu, f"self-loop edge {pv} -> {pu}"
        GGG.add_edge(pv, pu, **edge_data)

    edge_orientation: dict[frozenset, tuple[int, int]] = {}  # frozenset({a, b}) -> (src, dst)

    def orient(src: int, dst: int) -> bool:
        """Record src -> dst for this undirected edge; False if already oriented."""
        key = frozenset((src, dst))
        if key in edge_orientation:
            assert edge_orientation[key] == (src, dst), (
                f"conflicting orientation for edge {src}-{dst}: "
                f"already {edge_orientation[key]}, now requested {(src, dst)}"
            )
            return False
        edge_orientation[key] = (src, dst)
        return True

    def propagate(queue: deque) -> None:
        """Derive orientations from already-oriented incident edges (see note above)."""
        while queue:
            n = queue.popleft()
            neighbors = list(undirected.neighbors(n))
            known = {
                nb: edge_orientation[frozenset((n, nb))]
                for nb in neighbors
                if frozenset((n, nb)) in edge_orientation
            }
            if not known or len(known) == len(neighbors):
                continue  # nothing to derive yet, or already fully resolved

            if len(neighbors) == 2:
                kn, kd = next(iter(known.items()))
                un = next(nb for nb in neighbors if nb not in known)
                into_n = kd == (kn, n)
                if orient(*((un, n) if not into_n else (n, un))):
                    queue.append(un)

            elif len(neighbors) == 3:
                close_pair = close_neighbor_pairs[n]
                role = {nb: ("leg" if nb in close_pair else "point") for nb in neighbors}

                leg_into_n = next(
                    (d == (nb, n)) if role[nb] == "leg" else not (d == (nb, n))
                    for nb, d in known.items()
                )
                for nb in neighbors:
                    want_into_n = leg_into_n if role[nb] == "leg" else not leg_into_n
                    if orient(*((nb, n) if want_into_n else (n, nb))):
                        queue.append(nb)

            else:
                raise ValueError(f"unexpected degree {len(neighbors)} at node {n}")

    # pick any still-unoriented edge, orient it arbitrarily, and flood-fill the
    # rest of its connected component via propagate() - repeat until every edge
    # of GG has been oriented
    for a, b in undirected.edges:
        if frozenset((a, b)) not in edge_orientation:
            orient(a, b)
            propagate(deque([a, b]))

    assert len(edge_orientation) == undirected.number_of_edges(), "not all edges of GG were oriented"

    for src, dst in edge_orientation.values():
        add_oriented_edge(src, dst)

    return GGG


In [ ]:
GGG = build_ggg_impl2(GG, close_neighbor_pairs)


In [ ]:
# verify implementation 2 (general, order-independent edge-orientation construction)
verify_edge_duality(GGG)


In [ ]:
verify_ggg_paths(GGG, boundary_paths)


## 3rd implementation: single-pass BFS, close-pair check folded into both branches


In [ ]:
def build_ggg_impl3(GG, close_neighbor_pairs, G_proj):
    """3rd implementation: single-pass BFS (open/closed), generalizing the
    degree-2 and degree-3 handling into one rule: whichever already-wired
    neighbor of n is found first fixes n's established identity ("me"); for
    switches, that neighbor's leg/point group then also fixes the required
    role (into/out of me) for every other neighbor - keeping the two legs on
    the same side and the point on the opposite side. This close-pair check
    is folded into BOTH the "neighbor already wired elsewhere" (merge,
    postponed if not yet resolvable) and "neighbor still fresh" cases, so a
    switch's rule is honored no matter which of its neighbors gets discovered
    first - including a switch directly adjacent to another switch (e.g. a
    crossover).

    Caveat: unlike the propagate()-based 2nd implementation, this doesn't
    globally verify consistency as it goes, so a topologically-impossible
    junction (e.g. the 3-switch triangular loop discussed above) won't raise -
    it will just get resolved one particular way depending on BFS order, and
    the no-leg-to-leg guarantee may not hold for that specific junction while
    still holding everywhere else.
    """

    GGG = nx.DiGraph()
    GGG.graph["crs"] = GG.graph.get("crs")
    undirected = GG.to_undirected()

    def add_oriented_edge(u, v):
        """Add edge u -> v to GGG, plus its reverse-direction mirror prime(v) -> prime(u)."""
        u, v = str(u), str(v)
        u_id = int(u[:-len("_prime")]) if u.endswith("_prime") else int(u)
        v_id = int(v[:-len("_prime")]) if v.endswith("_prime") else int(v)
        edge_data = GG.get_edge_data(u_id, v_id) or GG.get_edge_data(v_id, u_id) or {}

        GGG.add_node(u, **GG.nodes[u_id])
        GGG.add_node(v, **GG.nodes[v_id])
        assert u != v, f"self-loop edge {u} -> {v}"
        GGG.add_edge(u, v, **edge_data)

        pu, pv = prime(u), prime(v)
        GGG.add_node(pv, **GG.nodes[v_id])
        GGG.add_node(pu, **GG.nodes[u_id])
        assert pv != pu, f"self-loop edge {pv} -> {pu}"
        GGG.add_edge(pv, pu, **edge_data)

    def find_me(en: str, n_str: str):
        """Find which of n_str/prime(n_str) is the identity of the current node
        already connected to `en` (or prime(en)) in GGG - regardless of edge
        direction (direction-agnostic same_variant/crossing check)."""
        en, n_str = str(en), str(n_str)
        same_variant = ((en, n_str) in GGG.edges or (n_str, en) in GGG.edges
                         or (prime(en), prime(n_str)) in GGG.edges or (prime(n_str), prime(en)) in GGG.edges)
        crossing = ((en, prime(n_str)) in GGG.edges or (prime(n_str), en) in GGG.edges
                    or (prime(en), n_str) in GGG.edges or (n_str, prime(en)) in GGG.edges)
        if same_variant and not crossing:
            return n_str
        elif crossing and not same_variant:
            return prime(n_str)
        elif same_variant and crossing:
            raise ValueError(f"both identities of {n_str} already connect to {en}/{prime(en)} - genuine conflict")
        else:
            raise ValueError(f"no edge found connecting {en}/{prime(en)} to {n_str}/{prime(n_str)}")

    def group_role(n, nb):
        """'leg' or 'point' for neighbor `nb` of switch `n`; None if n isn't a switch."""
        if n not in close_neighbor_pairs:
            return None
        return "leg" if nb in close_neighbor_pairs[n] else "point"

    def edge_into(target, source_candidate):
        """True if source_candidate (or its prime) already has an edge into
        `target` in GGG; False if `target` already has an edge out to it."""
        sc = str(source_candidate)
        if (sc, target) in GGG.edges or (prime(sc), target) in GGG.edges:
            return True
        elif (target, sc) in GGG.edges or (target, prime(sc)) in GGG.edges:
            return False
        else:
            raise ValueError(f"no edge found between {target} and {sc}/{prime(sc)}")

    def desired_into_me(n, me, known_neighbor, target):
        """Whether `target` should feed INTO `me` (True) or receive FROM `me`
        (False), given `known_neighbor` is n's already-established neighbor."""
        known_into_me = edge_into(me, known_neighbor)
        if n not in close_neighbor_pairs:
            return not known_into_me  # plain node: the other neighbor takes the opposite role
        same_group = group_role(n, known_neighbor) == group_role(n, target)
        return known_into_me if same_group else not known_into_me

    open_ = deque()

    initial = list(GG.nodes)[0]
    neighbors = list(undirected.neighbors(initial))
    if len(neighbors) == 2:
        add_oriented_edge(initial, neighbors[0])
        add_oriented_edge(neighbors[1], initial)
    elif len(neighbors) == 3:
        close_pair = close_neighbor_pairs[initial]
        point = next(x for x in neighbors if x not in close_pair)
        add_oriented_edge(point, initial)
        for leg in close_pair:
            # closest_neighbors() can hand back float-typed ids - normalize to int
            add_oriented_edge(initial, int(leg))
    else:
        raise ValueError(f"unexpected degree {len(neighbors)} at initial node {initial}")

    open_.extend(neighbors)
    closed = {initial}

    while open_:
        n = open_.popleft()
        if n in closed:
            continue

        neighbors = list(undirected.neighbors(n))
        ggg_undirected = GGG.to_undirected()
        existing = [nb for nb in neighbors
                    if ggg_undirected.has_edge(str(n), str(nb)) or ggg_undirected.has_edge(prime(str(n)), str(nb))]
        assert existing, f"node {n} visited with no existing GGG connection"

        known_neighbor = existing[0]
        me = find_me(str(known_neighbor), str(n))
        for other in existing[1:]:
            assert find_me(str(other), str(n)) == me, (
                f"node {n}'s existing neighbors resolve to different identities"
            )

        closed.add(n)
        postponed = False
        for nn in neighbors:
            if nn in existing:
                continue
            want_into_me = desired_into_me(n, me, known_neighbor, nn)

            if str(nn) in GGG.nodes:
                # nn already wired elsewhere - find which of its identities is
                # already resolved, via whichever of its other neighbors it's
                # already connected to
                nn_candidates = [str(x) for x in find_other_neighbor(undirected, nn, n, G_proj)]
                ggg_undirected = GGG.to_undirected()
                nn_in_ggg = [x for x in nn_candidates if ggg_undirected.has_edge(x, str(nn))]
                if not nn_in_ggg:
                    open_.append(n)
                    closed.remove(n)
                    postponed = True
                    break
                v = find_me(nn_in_ggg[0], str(nn))
                if want_into_me:
                    add_oriented_edge(v, me)
                else:
                    add_oriented_edge(me, v)
            else:
                if want_into_me:
                    add_oriented_edge(str(nn), me)
                else:
                    add_oriented_edge(me, str(nn))
                open_.append(nn)

    return GGG


In [ ]:
GGG = build_ggg_impl3(GG, close_neighbor_pairs, G_proj)


In [ ]:
# verify implementation 3 (single-pass BFS, close-pair folded into both branches)
verify_edge_duality(GGG)


In [ ]:
verify_ggg_paths(GGG, boundary_paths)


## Display GGG as interactive map

In [ ]:
# same categories as above (see "Visual inspection" / interactive map section)
explore_categorized(GGG)

In [ ]:
len(GGG.edges())

In [ ]:
for e in GGG.edges():
    print(e)

In [ ]:
import networkx as nx
nx.draw(GGG, labels={n[0]: str(n[0]) for n in list(GGG.nodes(data=True))})

## Results: which GGG-construction approach passes which verification

| Implementation | Graph | Completes? | `verify_edge_duality` | `verify_ggg_paths` |
|---|---|---|---|---|
| **1** — BFS, inline degree-based case analysis | original (`G`) | crashes (`ValueError`, no edge found) | — | — |
| **1** | `G_fixed` | yes | pass | **fail** |
| **2** — global `orient`/`propagate` flood-fill | original (`G`) | crashes (`AssertionError` at the [triangular junction](https://www.openstreetmap.org/node/2042222928)) | — | — |
| **2** | `G_fixed` | yes | pass | **pass** |
| **3** — single-pass BFS, close-pair check folded into both branches | original (`G`) | yes | pass | fail (only at the [triangular junction](https://www.openstreetmap.org/node/2042222928)) |
| **3** | `G_fixed` | yes | pass | **pass** |

### Why "locality" is not enough

Implementations 1-3 all decide a node's two `GGG` identities (plain / `_prime`) using only *locally* available
information: the node's immediate neighbors, and - for switches - which neighbor pair is the close (leg) pair.
This is enough almost everywhere, because the doubling scheme only has one degree of freedom per plain/undirected
edge (which of its two directed orientations is "into" vs "out of" a given node), and propagating that choice
along a path of degree-2 nodes and ordinary switches never conflicts with itself.

It breaks down on an **odd cycle of switches**: three switches connected pairwise (e.g. the triangular/wye
junction at [2042222928](https://www.openstreetmap.org/node/2042222928), switches
[2042222934](https://www.openstreetmap.org/node/2042222934),
[2042222937](https://www.openstreetmap.org/node/2042222937),
[2042222947](https://www.openstreetmap.org/node/2042222947)). Walking around the loop, each switch "flips" which
of its two neighbors is the leg-into vs. leg-out-of side relative to the point. An odd number of flips around a
closed loop is self-contradictory (same parity problem as 2-coloring an odd cycle) - **no choice of local
propagation order, and no amount of backtracking, can satisfy all three switches' leg/point constraints
simultaneously** if all three are mutually adjacent with no node in between. This is a structural property of the
graph topology, not a bug in any one implementation - it's why implementation 2 crashes loudly (it checks global
consistency and asserts), why implementation 3 silently produces an inconsistent path through that junction (it
never cross-checks consistency globally, only locally during its single BFS pass), and why implementation 1 never
even gets that far (it crashes earlier on an unrelated, still-unfixed gap in its own degree-3 case analysis that
ignores close-pairs for the "1 existing neighbor" branch).

### Why we need to fix the switch-switch edges

The odd cycle only exists because OSMnx's default `simplify_graph` collapses every interstitial (non-junction)
node on a way, including between two adjacent switches - leaving the three switches of the triangular junction
directly pairwise adjacent, with no "neutral" node between any two of them to absorb the parity conflict. Once a
single intermediate node is restored on just one of the triangle's three switch-switch edges (see
`simplify_graph_keep_switch_intermediate` above, applied to the non-simplified graph `G_non_simple`), that edge is
no longer a direct switch-to-switch edge - the loop is broken, the parity constraint disappears, and both
implementation 2 and implementation 3 pass both verifications on `G_fixed`. This is therefore a **data-level**
fix (recovering OSM information that `simplify_graph` is discarding), not an algorithmic one - no purely
algorithmic fix to the construction logic can resolve a genuine odd cycle of mutually-adjacent switches.

Implementation 1's remaining failure on `G_fixed` is unrelated to the triangular junction; it is the pre-existing
gap in its degree-3/"1 existing neighbor" branch, which never checks close-neighbor pairs (unlike implementations
2 and 3) and has not been fixed.


In [ ]:
# add actions to GGG (cf. "# add actions" for GG above) such that at every node, each rail env action is either
# prohibited or in the actions of exactly one outgoing edge (verified by GraphTransitionMap._assert_actions_partitioned()
# after creating the env below): at a dead end, all actions are prohibited; a single outgoing edge (non-switch, or
# switch passed from a leg into its point) gets all actions; at a switch passed from its point into its two legs (its
# close neighbors, see closest_neighbors()), the diverging leg gets MOVE_RIGHT (right switch) or MOVE_LEFT (left
# switch) and the forward leg all other actions (see switch_legs())
undirected = GG.to_undirected()
for n in GGG.nodes:
    successors = list(GGG.successors(n))
    if len(successors) == 0:
        GGG.nodes[n]["prohibited_actions"] = set(RailEnvActions)
        continue
    if len(successors) == 1:
        GGG.edges[n, successors[0]]["actions"] = set(RailEnvActions)
        continue
    # GGG node "x" / "x_prime" is GG node int(x)
    switch = int(n.removesuffix("_prime"))
    legs = {int(l.removesuffix("_prime")): l for l in successors}
    assert set(legs) == {int(x) for x in close_neighbor_pairs[switch]}, (
        f"outgoing edges of switch {n} don't lead to its close neighbors"
    )
    (point,) = set(undirected.neighbors(switch)) - set(legs)
    forward, diverging, diverges_right = switch_legs(G_proj, switch, point)
    turn = RailEnvActions.MOVE_RIGHT if diverges_right else RailEnvActions.MOVE_LEFT
    GGG.edges[n, legs[forward]]["actions"] = set(RailEnvActions) - {turn}
    GGG.edges[n, legs[diverging]]["actions"] = {turn}
    print(n, {l: GGG.edges[n, l]["actions"] for l in successors})

In [ ]:
from flatland.envs.observations import FullEnvObservation

# GGG nodes are str(n) / f"{n}_prime" (not GG's int nodes). Longest switch-to-switch
# shortest path in GGG: 14 edges, through 7 other switches
# (2042222936, 2042193124, 2048050594, 2048050596, 2048248467, 2048248475, 2048248546)
SWITCH = "2042222934"  # switch of the triangular junction
TARGET = "2048248489"  # switch
assert nx.has_path(GGG, SWITCH, TARGET), f"no directed path {SWITCH} -> {TARGET} in GGG"
print(nx.shortest_path(GGG, SWITCH, TARGET))

resource_map = {}
for n1, n2 in GGG.edges:
    resource_map[n1] = str(sorted([n1,n2]))
# print(resource_map)

agent_waypoints = {
    0: [[SWITCH], [TARGET]]
}
timetable = Timetable(
    earliest_departures=[[0,None]],
    # edge lengths are osmnx's `length` (metres): the 14 edges sum to ~3760 m, which take ~7520 steps at speed 1/2
    latest_arrivals=[[None,8000]],
    max_episode_steps=10000,
)

agent_speeds = {
    0: 1/2
}

graph_env = GraphRailEnv.from_graph(
    g=GGG,
    resource_map=resource_map,
    agent_waypoints=agent_waypoints,
    agent_speeds=agent_speeds,
    # GraphShortestPathPolicy below needs the env as observation
    observation_builder=FullEnvObservation(),
    malfunction_generator=mal_gen.NoMalfunctionGen(),
    timetable_generator=lambda *args, **kwargs: timetable,
    seed=42,
    rewards=DefaultRewards(),
)
graph_env.rail._assert_actions_partitioned()
graph_env.reset()

## Shortest-path policy on GGG

In [ ]:
from typing import List

from flatland.envs.agent_utils import EnvAgent
from flatland.envs.rail_env_policy import RailEnvPolicy
from flatland.envs.step_utils.states import TrainState


class GraphShortestPathPolicy(RailEnvPolicy[GraphRailEnv, GraphRailEnv, RailEnvActions]):
    """Follow the shortest path (in number of edges) in `env.rail.g` from the agent's
    initial entry point to its first reachable target - cf. `ShortestPathPolicy` in
    flatland/envs/rail_env_policies.py, but on the graph instead of the grid.
    Needs `FullEnvObservation` (observations are the env itself)."""

    def __init__(self):
        super().__init__()
        self._shortest_paths = {}

    def _act(self, env: GraphRailEnv, agent: EnvAgent):
        if len(self._shortest_paths[agent.handle]) <= 1:
            return RailEnvActions.DO_NOTHING

        if agent.current_entry_point is None:
            # actions are applied at cell entry: the action entering the map already decides
            # the transition out of the initial entry point (a switch, possibly)
            lookahead_from = agent.initial_entry_point
            target_index = 1
        else:
            # once agent.next_entry_point holds a pending target (decided one step ago), this
            # step's action decides the look-ahead beyond it, so evaluate from
            # next_entry_point against the node after it
            has_pending_target = agent.next_entry_point is not None and agent.next_entry_point != agent.current_entry_point
            lookahead_from = agent.next_entry_point if has_pending_target else agent.current_entry_point
            target_index = 2 if has_pending_target else 1
        if target_index >= len(self._shortest_paths[agent.handle]):
            return RailEnvActions.MOVE_FORWARD

        next_entry_point = self._shortest_paths[agent.handle][target_index]
        for a in [RailEnvActions.MOVE_FORWARD, RailEnvActions.MOVE_LEFT, RailEnvActions.MOVE_RIGHT]:
            if env.rail.apply_action_independent(a, lookahead_from) == next_entry_point:
                return a
        raise Exception(f"Invalid state: no action leads from {lookahead_from} to {next_entry_point}")

    def act_many(self, handles: List[int], observations: List[GraphRailEnv], **kwargs):
        actions = {}
        for handle, env in zip(handles, observations):
            agent = env.agents[handle]
            self._update_agent(agent, env)
            actions[handle] = self._act(env, agent)
        return actions

    def _update_agent(self, agent: EnvAgent, env: GraphRailEnv):
        """
        Update `_shortest_paths`.
        """
        if agent.state == TrainState.DONE:
            self._shortest_paths.pop(agent.handle, None)
            return

        if agent.handle not in self._shortest_paths:
            p = [agent.waypoints[0][0]]
            for pp2 in agent.waypoints[1:]:
                p_next = None
                for p2 in pp2:
                    if nx.has_path(env.rail.g, p[-1], p2):
                        p_next = nx.shortest_path(env.rail.g, p[-1], p2)[1:]
                        break
                assert p_next is not None, f"Not found next path from {p[-1]} to any of {pp2}, agent.waypoints={agent.waypoints}"
                p += p_next
            self._shortest_paths[agent.handle] = p

        if agent.current_entry_point is None:
            return

        while self._shortest_paths[agent.handle][0] != agent.current_entry_point:
            self._shortest_paths[agent.handle] = self._shortest_paths[agent.handle][1:]


In [ ]:
import shutil
from pathlib import Path

from flatland.trajectories.policy_runner import PolicyRunner

data_dir = Path("trajectories")
# start from an empty dir: flatland can't read back graph positions from an existing
# trajectory (pandas parses node ids like 2042222934 as floats, which
# normalize_position_read() can't literal_eval)
shutil.rmtree(data_dir / "shortest_path", ignore_errors=True)
# GraphRailEnv does not support env persistence yet - hence no_save/no snapshots
trajectory = PolicyRunner.create_from_policy(
    env=graph_env,
    policy=GraphShortestPathPolicy(),
    data_dir=data_dir / "shortest_path",
    no_save=True,
    snapshot_interval=0,
)
display(trajectory.trains_positions)
print(graph_env.agents[0].state, graph_env.agents[0].current_entry_point)
assert graph_env.agents[0].state == TrainState.DONE, f"train did not arrive: {graph_env.agents[0].state}"


## Step through the episode on the map

In [ ]:
# !python -m pip install "ipyleaflet>=0.19"
# step through an episode with GraphShortestPathPolicy, showing the train on the map:
# each train is an OSM-like feature tagged historic=locomotive, drawn with that tag's
# icon from the OSM wiki - N.B. the tag is only interpreted here (client-side), the
# rendered OSM tiles know nothing about it.
# The map (ipyleaflet) is created once and only its layers are updated on each step,
# so zoom/pan are kept.
import math
import shutil

import ipywidgets as widgets
from ipyleaflet import CircleMarker, Icon, LayerGroup, Map, Marker, Polyline
from IPython.display import display

from flatland.trajectories.trajectories import Trajectory

# OSM tag -> icon (https://wiki.openstreetmap.org/wiki/File:Historic_locomotive.png, CC BY-SA)
TAG_ICONS = {
    ("historic", "locomotive"): "https://upload.wikimedia.org/wikipedia/commons/e/ef/Historic_locomotive.png",
}


def train_features(env: GraphRailEnv) -> list[dict]:
    """One OSM-like feature (tags + location) per train, location None if not on the map."""
    return [
        {
            "handle": agent.handle,
            "tags": {"historic": "locomotive", "name": f"train {agent.handle}"},
            "location": (env.rail.g.nodes[c]["y"], env.rail.g.nodes[c]["x"]) if (c := agent.current_entry_point) is not None else None,
            "title": f"train {agent.handle}: {c} ({agent.state.name})",
        }
        for agent in env.agents
    ]


def edge_locations(g: nx.DiGraph, u: str, v: str) -> list[tuple[float, float]]:
    """(lat, lon) points along edge (u, v) - its geometry if any, else the straight segment."""
    geometry = g.edges[u, v].get("geometry")
    if geometry is not None:
        return [(y, x) for x, y in geometry.coords]
    return [(g.nodes[n]["y"], g.nodes[n]["x"]) for n in (u, v)]


def gg_node(n: str) -> int:
    """GG node of GGG node "x" / "x_prime"."""
    return int(n.removesuffix("_prime"))


def create_map(g: nx.DiGraph) -> Map:
    """Static layers of GGG: edges green, nodes colored by switch category (cf.
    explore_categorized()) - plain/_prime twins share their geometry, draw them once."""
    track = {frozenset((gg_node(u), gg_node(v))): (u, v) for u, v in g.edges}
    nodes = {gg_node(n): n for n in g.nodes}
    # fixed size, so that fit_to_bounds() can compute the zoom level fitting the graph
    m = Map(scroll_wheel_zoom=True, layout=widgets.Layout(width=f"{MAP_WIDTH}px", height=f"{MAP_HEIGHT}px"))
    m.add(Polyline(locations=[edge_locations(g, u, v) for u, v in track.values()], color="green", fill=False, weight=3))
    m.add(LayerGroup(layers=[
        CircleMarker(
            location=(g.nodes[n]["y"], g.nodes[n]["x"]), radius=mk["radius"], weight=1, fill_opacity=0.8,
            color=(low_degree_switch_color if osmid in switch_attr_low_degree_set
                   else switch_color if osmid in switches_set else default_color),
        )
        for osmid, n in nodes.items()
    ]))
    return m


def graph_bounds(g: nx.DiGraph) -> list[list[float]]:
    """[[south, west], [north, east]] of all nodes of `g`."""
    lats = [g.nodes[n]["y"] for n in g.nodes]
    lons = [g.nodes[n]["x"] for n in g.nodes]
    return [[min(lats), min(lons)], [max(lats), max(lons)]]


# N.B. not Map.fit_bounds(): it zooms out/in step by step comparing against Map.bounds, which the rendered map
# reports asynchronously - stale bounds make it zoom out far too much (e.g. to all of Europe). Instead, compute the
# largest Web Mercator zoom level at which `bounds` fit into the map's fixed pixel size (256px tiles).
MAP_WIDTH, MAP_HEIGHT = 800, 600


def fit_to_bounds(m: Map, bounds: list[list[float]]):
    """Center `m` on `bounds` ([[south, west], [north, east]]) and zoom in as far as they still fit."""
    (south, west), (north, east) = bounds

    def mercator_y(lat: float) -> float:
        sin = math.sin(math.radians(lat))
        return math.log((1 + sin) / (1 - sin)) / 2

    lat_fraction = (mercator_y(north) - mercator_y(south)) / (2 * math.pi)
    lon_fraction = (east - west) / 360
    min_zoom = 0 if m.min_zoom is None else m.min_zoom
    max_zoom = 18 if m.max_zoom is None else m.max_zoom
    zooms = [math.log2(MAP_HEIGHT / 256 / lat_fraction) if lat_fraction > 0 else max_zoom,
             math.log2(MAP_WIDTH / 256 / lon_fraction) if lon_fraction > 0 else max_zoom]
    m.center = ((south + north) / 2, (west + east) / 2)
    m.zoom = max(min_zoom, min(max_zoom, math.floor(min(zooms))))


# for visualization, step through the episode on a copy of GGG with all edge lengths divided by the shortest
# edge's length (so the shortest edge has length 1) - with lengths in metres, it would take thousands of steps
shortest_edge_length = min(length for _, _, length in GGG.edges(data="length"))
GGG_step = GGG.copy()
for _, _, edge_data in GGG_step.edges(data=True):
    edge_data["length"] = edge_data["length"] / shortest_edge_length
step_env = GraphRailEnv.from_graph(
    g=GGG_step,
    resource_map=resource_map,
    agent_waypoints=agent_waypoints,
    agent_speeds=agent_speeds,
    observation_builder=FullEnvObservation(),
    malfunction_generator=mal_gen.NoMalfunctionGen(),
    timetable_generator=lambda *args, **kwargs: timetable,
    seed=42,
    rewards=DefaultRewards(),
)
step_env.rail._assert_actions_partitioned()

# start from an empty dir (see above)
shutil.rmtree(data_dir / "step", ignore_errors=True)
step_env.reset()
step_policy = GraphShortestPathPolicy()
step_runner = PolicyRunner(
    policy=step_policy,
    trajectory=Trajectory.create_empty(data_dir=data_dir / "step", env=None),
    by_pass_env=step_env,
)

step_map = create_map(step_env.rail.g)
# remaining shortest path per train, and the trains themselves - updated in place by show()
path_layers = {agent.handle: Polyline(locations=[], color="orange", fill=False, weight=6, opacity=0.8) for agent in step_env.agents}
train_markers = {}
for feature in train_features(step_env):
    icon_url = next(url for (k, v), url in TAG_ICONS.items() if feature["tags"].get(k) == v)
    train_markers[feature["handle"]] = Marker(
        location=feature["location"] or step_map.center, visible=False, draggable=False,
        icon=Icon(icon_url=icon_url, icon_size=[30, 30], icon_anchor=[15, 15]),
    )
for layer in [*path_layers.values(), *train_markers.values()]:
    step_map.add(layer)

step_button = widgets.Button(description="Step", icon="step-forward")
step_label = widgets.Label()


def show(fit_bounds: bool = False):
    """Update label, remaining paths and train markers; with `fit_bounds`, also zoom the map to the graph."""
    agent = step_env.agents[0]
    step_label.value = f"step {step_env._elapsed_steps}: {agent.state.name} at {agent.current_entry_point}"
    for handle, path_layer in path_layers.items():
        # the policy only drops a train's path on its next act_many() after DONE
        path = [] if step_env.agents[handle].state == TrainState.DONE else step_policy._shortest_paths.get(handle, [])
        path_layer.locations = [edge_locations(step_env.rail.g, u, v) for u, v in zip(path, path[1:])]
    for feature in train_features(step_env):
        marker = train_markers[feature["handle"]]
        marker.visible = feature["location"] is not None
        if feature["location"] is not None:
            marker.location = feature["location"]
        marker.title = feature["title"]
    if fit_bounds:
        fit_to_bounds(step_map, graph_bounds(step_env.rail.g))


def on_step(_):
    _, done = step_runner.step()
    step_button.disabled = done
    show()


step_button.on_click(on_step)
display(widgets.VBox([widgets.HBox([step_button, step_label]), step_map]))
# fit only initially - on_step()'s show() keeps the user's zoom/pan
show(fit_bounds=True)
